# 📓 04: Word2Vec Embeddings & Deep Learning Classifiers
**Hackathon Track:** SAMATRIX RESUMEFORGE 2026  
**Team:** NueralX  
**Author:** Savan (Person 3 — Branch: `savan_p3`)  
**Tasks Covered:** `P3.1` (Word2Vec), `P3.2` (Doc Pooling), `P3.3` (Neural Classifiers), `P3.4` (Validation & Early Stopping), `P3.5` (Evaluation), `P3.8` (Model Artifacts)

---

## 🎯 Objectives
1. **P3.1 Word2Vec Training:** Train custom continuous word embeddings **strictly on the training split** to guarantee zero data leakage.
2. **P3.2 Document Representation:** Aggregate word vectors into fixed-size document vectors via **Mean Pooling** (and compare with Max Pooling).
3. **P3.3 Neural Architecture Design:** Build two deep learning classifiers:
   - **Architecture A:** Multi-Layer Perceptron (Dense NN with ReLU, Dropout, Batching)
   - **Architecture B:** Bidirectional LSTM (BiLSTM) capturing sequential resume context
4. **P3.4 Validation Strategy:** Track validation loss and Macro-F1 curves; apply **early stopping** to prevent overfitting.
5. **P3.5 Rigorous Evaluation:** Evaluate models on the primary metric (**Macro-F1**) across all 24 industry categories.
6. **P3.8 Artifact Persistence:** Save trained embeddings, models, and label encoders for production deployment in Streamlit.

In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# Ensure project root is in sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.preprocessing import clean_resume_text, tokenize_resume
from src.word2vec import (
    train_word2vec_pytorch,
    get_document_vectors_pytorch,
    save_word2vec_artifacts,
    load_word2vec_artifacts
)
from src.train_dl import train_dense_nn, train_lstm_classifier, save_dl_artifacts

# Set seeds for exact reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

## 1. Load Preprocessed Stratified Splits
We use the exact stratified splits generated in Phase 1 (70% Train, 15% Validation, 15% Test) to ensure fairness across all 24 classes.

In [ ]:
train_path = os.path.join('..', 'data', 'processed', 'train.csv')
val_path = os.path.join('..', 'data', 'processed', 'val.csv')
test_path = os.path.join('..', 'data', 'processed', 'test.csv')

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)
test_df = pd.read_csv(test_path)

print(f"Training set:   {len(train_df):,} resumes")
print(f"Validation set: {len(val_df):,} resumes")
print(f"Test set:       {len(test_df):,} resumes")
print(f"Total classes:  {train_df['Category'].nunique()} categories")

# Ensure cleaned_text exists in each split
for df in [train_df, val_df, test_df]:
    if 'cleaned_text' not in df.columns:
        df['cleaned_text'] = df['Resume_str'].apply(clean_resume_text)

print("Sample cleaned resume text:")
print(train_df['cleaned_text'].iloc[0][:300] + "...")

## 2. Encode Target Categories & Tokenize
Fit a `LabelEncoder` on the training labels and map string categories to integer IDs.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train = le.fit_transform(train_df['Category'])
y_val = le.transform(val_df['Category'])
y_test = le.transform(test_df['Category'])

num_classes = len(le.classes_)
print(f"Fitted {num_classes} classes: {list(le.classes_[:5])}... and {num_classes-5} more")

# Tokenize documents
print("Tokenizing documents...")
train_tokens = [tokenize_resume(t) for t in train_df['cleaned_text']]
val_tokens = [tokenize_resume(t) for t in val_df['cleaned_text']]
test_tokens = [tokenize_resume(t) for t in test_df['cleaned_text']]

doc_lengths = [len(t) for t in train_tokens]
print(f"Mean doc length: {np.mean(doc_lengths):.1f} tokens, Median: {np.median(doc_lengths):.0f} tokens")

## 3. P3.1: Train Word2Vec Embeddings on Training Split Only
**Data Leakage Prevention:** Word embeddings are trained strictly on `train_tokens`.
No validation or test documents are included in vocabulary building or co-occurrence learning.

In [ ]:
w2v_model_path = os.path.join('..', 'models', 'word2vec_pytorch.pt')

if os.path.exists(w2v_model_path):
    print("Loading existing Word2Vec model from models/...")
    w2v_model, vocab = load_word2vec_artifacts(save_dir=os.path.join('..', 'models'))
    print(f"Loaded Word2Vec model with vocabulary size {len(vocab):,}")
else:
    print("Training PyTorch Skip-Gram Word2Vec on training corpus...")
    w2v_model, vocab = train_word2vec_pytorch(
        train_tokens,
        vector_size=150,
        window=5,
        min_count=2,
        epochs=10,
        batch_size=512,
        lr=0.003,
    )
    save_word2vec_artifacts(w2v_model, vocab, save_dir=os.path.join('..', 'models'))

## 4. P3.2: Document Representations via Pooling
Convert sequence of word embeddings into a fixed document-level feature vector using **mean pooling**.

In [ ]:
print("Generating document vectors via mean pooling...")
X_train_emb = get_document_vectors_pytorch(train_tokens, w2v_model, vocab, pooling="mean")
X_val_emb = get_document_vectors_pytorch(val_tokens, w2v_model, vocab, pooling="mean")
X_test_emb = get_document_vectors_pytorch(test_tokens, w2v_model, vocab, pooling="mean")

print(f"X_train shape: {X_train_emb.shape}")
print(f"X_val shape:   {X_val_emb.shape}")
print(f"X_test shape:  {X_test_emb.shape}")

## 5. P3.3 & P3.4: Train Neural Classifiers with Early Stopping
We evaluate two distinct neural architectures:
1. **Dense NN (Multi-Layer Perceptron):** 3-layer architecture (256 -> 128 -> 64) with adaptive Adam optimizer and early stopping.
2. **Bidirectional LSTM:** Sequential architecture processing word embeddings directly with recurrent gating and dropout.

In [ ]:
# Architecture A: Dense NN
print("Training Dense NN (MLP)...")
dense_results = train_dense_nn(
    X_train_emb, y_train,
    X_val_emb, y_val,
    hidden_layers=(256, 128, 64),
    max_iter=100,
    lr=1e-3,
)

# Architecture B: BiLSTM
print("\nTraining Bidirectional LSTM...")
lstm_results = train_lstm_classifier(
    train_tokens, y_train,
    val_tokens, y_val,
    w2v_model, vocab,
    num_classes=num_classes,
    embed_dim=150,
    hidden_dim=128,
    max_len=300,
    epochs=20,
    batch_size=32,
    lr=1e-3,
)

## 6. P3.5: DL Model Comparison & Evaluation Metrics
Compare Dense NN vs BiLSTM on Validation Accuracy, Macro-F1 (Primary), and Weighted-F1.

In [ ]:
comparison_df = pd.DataFrame([
    {
        'Model': 'Dense NN (MLP)',
        'Accuracy': dense_results['metrics']['accuracy'],
        'Macro-Precision': dense_results['metrics']['macro_precision'],
        'Macro-Recall': dense_results['metrics']['macro_recall'],
        'Macro-F1 (Primary)': dense_results['metrics']['macro_f1'],
        'Weighted-F1': dense_results['metrics']['weighted_f1'],
    },
    {
        'Model': 'Bidirectional LSTM',
        'Accuracy': lstm_results['metrics']['accuracy'],
        'Macro-Precision': lstm_results['metrics']['macro_precision'],
        'Macro-Recall': lstm_results['metrics']['macro_recall'],
        'Macro-F1 (Primary)': lstm_results['metrics']['macro_f1'],
        'Weighted-F1': lstm_results['metrics']['weighted_f1'],
    }
])

print("=" * 65)
print("           DEEP LEARNING MODEL BENCHMARK (VALIDATION SET)")
print("=" * 65)
print(comparison_df.to_string(index=False))
print("=" * 65)

## 7. Confusion Matrix Visualization
Analyze misclassifications across the 24 categories to identify overlapping domains (e.g., FINANCE vs BANKING, HR vs BPO).

In [ ]:
from sklearn.metrics import confusion_matrix

best_preds = dense_results['predictions'] if dense_results['metrics']['macro_f1'] >= lstm_results['metrics']['macro_f1'] else lstm_results['predictions']
cm = confusion_matrix(y_val, best_preds)

plt.figure(figsize=(14, 11))
sns.heatmap(
    cm,
    annot=False,
    fmt='d',
    cmap='Blues',
    xticklabels=le.classes_,
    yticklabels=le.classes_,
)
plt.title('Validation Confusion Matrix — Best Deep Learning Model', fontsize=14, pad=15)
plt.xlabel('Predicted Category', fontsize=12)
plt.ylabel('True Category', fontsize=12)
plt.xticks(rotation=90, fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('../models/dl_confusion_matrix.png', dpi=300)
plt.show()
print("Confusion matrix saved to models/dl_confusion_matrix.png")

## 8. P3.8: Save Production Model Artifacts
Persist all artifacts to `../models/` for seamless integration into the Streamlit application.

In [ ]:
save_dir = os.path.join('..', 'models')
best_type = 'dense_nn' if dense_results['metrics']['macro_f1'] >= lstm_results['metrics']['macro_f1'] else 'lstm'
best_model = dense_results['model'] if best_type == 'dense_nn' else lstm_results['model']

save_dl_artifacts(
    w2v_model=w2v_model,
    vocab=vocab,
    classifier=best_model,
    label_encoder=le,
    classifier_type=best_type,
    save_dir=save_dir,
)

# Ensure dense_nn is always saved as fallback for streamlit
joblib.dump(dense_results['model'], os.path.join(save_dir, 'dense_nn_classifier.joblib'))
joblib.dump(le, os.path.join(save_dir, 'label_encoder.joblib'))
print("All production artifacts saved successfully!")